# 01 — Load Data

**Purpose:** Load all datasets and confirm everything is present and properly formatted before any analysis.

This notebook does **no modeling**. It just verifies the data pipeline is intact:
- Session configuration
- Rokoko motion capture data
- EEG recording
- ELAN sign annotations
- BioSemi trigger markers

In [ ]:
import sys, os

# Set working directory to project root so all relative paths work
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'Project root: {ROOT}')

## 1. Session Configuration

The session config (`config/session.yaml`) stores all recording-specific facts:
participant, sampling rate, channel list, file paths, and trigger codes.
Changing participant or recording only requires updating this one file.

In [ ]:
from sign_language_trf.config import load_session_config

cfg = load_session_config('config/session.yaml')
print(f'Participant     : {cfg.participant}')
print(f'EEG sample rate : {cfg.srate} Hz  →  analysis rate: {cfg.common_rate} Hz')
print(f'Presentations   : {cfg.n_presentations}')
print(f'Drop channels   : {cfg.drop_channels}')
print(f'EEG path        : {cfg.eeg_path}')
print(f'Motion path     : {cfg.motion_path}')

## 2. Motion Data (Rokoko)

The Rokoko suit records Tristan's body position at ~58.82 Hz.
The processed CSV contains per-joint velocities and combined motion energy.
This is the source for all kinematic regressors.

In [ ]:
from sign_language_trf.preprocessing.motion_loader import load_motion

motion = load_motion(cfg.motion_path)
print(f'Duration      : {motion.duration_s:.1f} s')
print(f'Sample rate   : {motion.fs:.2f} Hz')
print(f'N samples     : {len(motion.time_s)}')
print(f'Participant   : {motion.participant}')
print(f'\nAvailable columns:')
import pandas as pd
df = pd.read_csv(cfg.motion_path)
for col in df.columns:
    print(f'  {col}')

## 3. EEG Data

Recorded with BioSemi at 512 Hz, 35 channels, ICA-cleaned.
6 non-scalp channels are dropped (mastoids, eye, and confirmed bad channels).
A 0.1–30 Hz bandpass is applied before resampling to 64 Hz.

In [ ]:
from sign_language_trf.preprocessing.eeg_loader import load_eeg

eeg = load_eeg(cfg.eeg_path, cfg, bandpass=True, l_freq=0.1, h_freq=30.0, verbose=False)
print(f'Participant   : {eeg.participant}')
print(f'Duration      : {eeg.duration_s:.1f} s')
print(f'Sample rate   : {eeg.sfreq} Hz (raw: {eeg.sfreq_raw} Hz)')
print(f'N channels    : {len(eeg.ch_names)}')
print(f'Channels      : {eeg.ch_names}')
print(f'\nEvents:')
import numpy as np
codes, counts = np.unique(eeg.events[:, 2], return_counts=True)
for code, count in zip(codes, counts):
    label = {255: 'video onset', 254: 'video end', 50: 'sync pulse'}.get(int(code), 'other')
    print(f'  Code {code:>3}: {count:>4}x  ({label})')

## 4. ELAN Annotations

ELAN annotations from the Gloss tier mark the start and end time of each sign
Tristan produces. These include lexical signs, sub-lexical morphemes, and
classifiers — not just whole words.

We use these to define linguistically-grounded time windows for regressor extraction.

In [ ]:
from sign_language_trf.preprocessing.elan_loader import load_elan
import glob

elan_files = glob.glob('sign_language_trf/data/elan/*.eaf')
print(f'ELAN file: {elan_files[0]}')
elan = load_elan(elan_files[0])

print(f'\nTier          : {elan.tier_id}')
print(f'N annotations : {elan.n_annotations}')
print(f'Signing window: {elan.signing_onset_s:.1f} – {elan.signing_offset_s:.1f} s')
print(f'\nFirst 10 signs:')
for ann in elan.annotations[:10]:
    print(f'  {ann.onset_s:7.3f}s – {ann.offset_s:7.3f}s  ({ann.offset_s - ann.onset_s:.3f}s)  "{ann.gloss}"')

## 5. Trigger Markers

The BioSemi marker CSV records exactly when each video presentation started
according to the Rokoko clock. This is used to align motion data with EEG.

In [ ]:
from sign_language_trf.preprocessing.markers_loader import load_markers

markers = load_markers(cfg.markers_path)
print(f'N markers     : {len(markers.times_s)}')
print(f'Marker times  : {markers.times_s[:10]} ...')
print(f'Median interval: {np.median(np.diff(markers.times_s)) * 1000:.1f} ms')

## Summary

If all cells above ran without errors, all data is present and correctly formatted.
Proceed to `02_preprocessing.ipynb` to generate regressors.

In [ ]:
print('=== Data Summary ===')
print(f'Motion  : {motion.duration_s:.0f}s at {motion.fs:.1f} Hz')
print(f'EEG     : {eeg.duration_s:.0f}s, {len(eeg.ch_names)} channels at {eeg.sfreq} Hz')
print(f'ELAN    : {elan.n_annotations} signs, {elan.signing_onset_s:.0f}–{elan.signing_offset_s:.0f}s')
print(f'Markers : {len(markers.times_s)} sync pulses')
print('\nAll data loaded successfully.')